# 🧠ИИ-ассистент “ТехНадзор”

## Шаг 1. Импорты и базовые настройки

In [ ]:
import os
import re
import pickle
import PyPDF2

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_gigachat import GigaChat  
from langchain.schema import Document


📦 Описание библиотек:

- **langchain-community** — предоставляет готовые компоненты, такие как FAISS, для работы с векторными базами данных.
- **langchain-huggingface** — позволяет использовать модели из Hugging Face- для создания эмбеддингов текста.
- **langchain** — основной фреймворк для объединения разных компонентов (моделей, промтов, ретриверов) в единые цепочки.
- **langchain-gigachat** — интеграция, которая позволяет использовать языковую модель GigaChat внутри фреймворка LangChain.
- **PyPDF2** — библиотека для чтения и извлечения текста из PDF-файлов.
- **os** — стандартный модуль Python для взаимодействия с операционной системой, например, для проверки наличия файлов.
- **re** — модуль для работы с регулярными выражениями, используемый для очистки и обработки текста.
- **pickle** — модуль для сохранения и загрузки объектов Python (в данном случае, документов и списка пунктов) в файлы.
- **telegram.ext** — фреймворк для создания Telegram-ботов, который упрощает обработку команд и сообщений от пользователей.


## Шаг 2. Пути к данным и список PDF-файлов

In [ ]:
INDEX_PATH = "faiss_index"
DOCS_PATH = "documents.pkl"

pdf_files = [
    ("Руководство по проектированию ЖБК 1977 года.pdf", 1977),
    ("Руководство по проектированию ЖБК 1978 года.pdf", 1978),
]


Указываем имена файлов и год для каждого документа.

`INDEX_PATH` и `DOCS_PATH` нужны, чтобы не пересоздавать индекс каждый раз.

## Шаг 3. Функция извлечения текста из PDF с метаданными

In [ ]:
def extract_text_with_metadata(path, year):
    docs = []
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        text_all = ""
        for i, page in enumerate(reader.pages, start=1):
            text = page.extract_text()
            if text:
                text = re.sub(r"-\n", "", text)   # убираем переносы по дефису
                text = re.sub(r"\n+", "\n", text) # нормализуем переносы
                text_all += f"\n<PAGE:{i}>\n" + text.strip()

    lines = text_all.split("\n")

    current_point = None
    buffer = []
    page_num = 1

    for line in lines:
        page_match = re.match(r"<PAGE:(\d+)>", line.strip())
        if page_match:
            page_num = int(page_match.group(1))
            continue

        m = re.match(r"^(\d+(?:\.\d+)+)\.\s*(.*)", line.strip())
        if m:
            if current_point and buffer:
                text_block = " ".join(buffer).strip()
                if len(text_block) > 30:
                    docs.append({
                        "text": f"{current_point}. {text_block}",
                        "source": f"Руководство {year}",
                        "point": current_point,
                        "page": page_num
                    })
            current_point = m.group(1)
            buffer = [m.group(2)]
        else:
            if buffer is not None:
                buffer.append(line.strip())

    if current_point and buffer:
        text_block = " ".join(buffer).strip()
        if len(text_block) > 30:
            docs.append({
                "text": f"{current_point}. {text_block}",
                "source": f"Руководство {year}",
                "point": current_point,
                "page": page_num
            })

    return docs


- Разбиваем PDF на **пункты** (по номерам вроде `3.4.`);
- Сохраняем текст + метаданные: год, пункт, страница.
- На выходе получаем список словарей с данными.


## Шаг 4. Настройка эмбеддингов (единая)

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)


Объяснение:

Выбираем **многоязычную модель эмбеддингов**. Она хорошо работает с русским.

## Шаг 5. Загрузка/создание индекса FAISS

In [ ]:
if os.path.exists(INDEX_PATH) and os.path.exists(DOCS_PATH):
    print("🔄 Загружаю готовый индекс...")
    with open(DOCS_PATH, "rb") as f:
        documents = pickle.load(f)
    vector_store = FAISS.load_local(INDEX_PATH, embedding, allow_dangerous_deserialization=True)

else:
    print("📄 Извлекаю текст из PDF...")
    documents = []
    for file, year in pdf_files:
        documents.extend(extract_text_with_metadata(file, year))

    print("📦 Строю FAISS...")
    docs_for_index = [
        Document(page_content=d["text"], metadata={"source": d["source"], "point": d["point"], "page": d["page"]})
        for d in documents
    ]

    vector_store = FAISS.from_documents(docs_for_index, embedding)
    vector_store.save_local(INDEX_PATH)

    with open(DOCS_PATH, "wb") as f:
        pickle.dump(docs_for_index, f)

print("✅ Индекс готов. Число пунктов:", len(documents))

Объяснение:

- Если индекс уже создан → загружаем;
- Если нет → читаем PDF, создаём список `Document` и строим новый FAISS.

## Шаг 6. Настройка LLM и промпта

In [ ]:
llm = GigaChat(
    model="GigaChat",
    credentials="ТОКЕН",  
    verify_ssl_certs=False
)

prompt = ChatPromptTemplate.from_template("""
Ты инженер-конструктор. Отвечай кратко и точно по содержанию руководств ЖБК 1977 и 1978 гг.
Если ответа нет — скажи: "Нет данных".

Формат:
- Ответ ≤15 слов.
- Укажи источник: (Руководство <год>, п. <номер>).
- Если из двух — укажи оба.

Контекст:
{context}

Вопрос: {input}

Краткий ответ:
""")


Объяснение:

- Настроили GigaChat;
- Задали строгий формат ответа (чтобы он был коротким и со ссылкой на пункт).

## Шаг 7. Функции для поиска и ответа

In [ ]:
def docs_formatter(docs):
    formatted = []
    for d in docs:
        src = d.metadata.get("source", "")
        point = d.metadata.get("point", "")
        page = d.metadata.get("page", "")
        formatted.append(f"[{src}, п.{point}, стр.{page}] {d.page_content}")
    return "\n".join(formatted)


document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
retriever = vector_store.as_retriever(search_kwargs={"k": 20})

def retrieval_chain_with_metadata(query: str):
    docs = retriever.get_relevant_documents(query)
    context = docs_formatter(docs)
    return document_chain.invoke({"context": context, "input": query})

def answer_question(query: str) -> str:
    response = retrieval_chain_with_metadata(query)
    return response


In [ ]:
#Тестирование:
answer_question("какой минимальный защитный слой бетона от арматуры?")

## Шаг 8. Интеграция с Telegram

In [ ]:
TELEGRAM_TOKEN = "Токен"

def start(update, context):
    update.message.reply_text("Привет! Я бот по железобетону. Задай мне вопрос, чтобы получить краткий ответ со всеми ссылками на нормативные документы")

def handle_message(update, context):
    user_q = update.message.text
    try:
        answer = answer_question(user_q)
    except Exception as e:
        answer = f"Ошибка: {e}"
    update.message.reply_text(answer)

updater = Updater(TELEGRAM_TOKEN, use_context=True)
dp = updater.dispatcher

dp.add_handler(CommandHandler("start", start))
dp.add_handler(MessageHandler(Filters.text & ~Filters.command, handle_message))

print("🚀 Бот запущен. Жду вопросы в Telegram...")
updater.start_polling()
updater.idle()
